### タスク 1.1: 基礎の設定とベーシックエージェントの作成

Strands Agents フレームワークを使用してカスタマーサポートエージェントのプロトタイプを設定します。このプロトタイプは、エージェントプロトタイプから本番環境対応ソリューションまでの全行程をチェックするための出発点となります。

このタスクを完了すると、エージェントは次の基本アーキテクチャを使用できるようになります。

<div style="text-align:left">
    <img src="images/architecture_lab1_strands_ja_jp.png" width="75%"/>
</div>

**画像の説明: ローカルツールでローカルで実行されるシンプルなエージェントプロトタイプ**

依存関係をインストールし、AWS SDK、AgentCore コンポーネント、Strands フレームワークなど必要なすべてのライブラリをインポートして、開発環境を準備します。

In [ ]:
import boto3
import json
import uuid
import time
import requests
from boto3.session import Session

# AgentCore imports
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

# Strands imports
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

# Local tools
from lab_helpers.lab1_strands_agent import (
    get_product_info, get_return_policy, get_technical_support, web_search,
    SYSTEM_PROMPT, MODEL_ID
)
from lab_helpers.utils import get_ssm_parameter, put_ssm_parameter
from scripts.utils import get_cognito_client_secret

# Setup
boto_session = Session()
REGION = "us-east-1"
CUSTOMER_ID = "customer_001"
SESSION_ID = str(uuid.uuid4())

print("✅ Libraries imported successfully!")

エージェントを作成する前に、カスタマーサポート機能を強化するローカルツールを調べます。`lab_helpers/lab1_strands_agent.py` を開いて確認し、以下を理解します。

- ツールは `@tool` デコレータを使用してこのファイル内でローカルに定義されます
- 4 つのツール機能とその目的:
  - get_product_info(): 商品情報を取得する
  - get_return_policy(): 特定の商品の返品ポリシーを取得する
  - get_technical_support(): テクニカルサポートガイダンスを提供する
  - web_search(): ウェブで最新情報を検索する
- モックデータの使用方法 (実際のデータベース / API のシミュレーション)
- エージェントの動作を定義するシステムプロンプト

クエリの理解からアクションの実行まで、AI のコア機能を実証するファウンデーショナルカスタマーサポートエージェントを作成します。このエージェントは以下を組み合わせたものです。

- **基盤モデル**: 推論と意思決定を支える「ブレイン」
- **システムプロンプト**: エージェントのパーソナリティとサービス基準を定義する動作指示
- **専門ツール**: 4 つのローカルツール (商品情報、返品ポリシー、テクニカルサポート、ウェブ検索)

エージェントを呼び出すと、エージェントは次のプロセスに従います。
1. **クエリ分析**: エージェントはお客様の質問を分析します
2. **ツールの選択**: エージェントは使用するツールを決定します (存在する場合)
3. **ツールの実行**: エージェントは適切なパラメータを使用して適切なツールを呼び出します
4. **レスポンス合成**: エージェントはツールの結果と知識を組み合わせて役立つレスポンスを作成します
5. **品質チェック**: エージェントはレスポンスがシステムプロンプトの基準を満たしていることを確認します

In [ ]:
# Create a basic agent with local tools
model = BedrockModel(model_id=MODEL_ID, temperature=0.3, region_name=REGION)
basic_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Basic customer support agent ready!")
print("📋 Available tools: Product Info, Return Policy, Technical Support, Web Search")

基本的なエージェントをテストして、お客様からの問い合わせをどのように処理し、ツールをどのように使用するかを確認します。

In [ ]:
# Test basic agent functionality
print("💬 Testing basic agent...\n")
response = basic_agent("What's the return policy for laptops?")
print("\n" + "="*50 + "\n")

この初期プロトタイプには、後続のタスクで対処するいくつかの制限があります。

- **永続的メモリなし** - エージェントが以前のセッションの顧客履歴や好みを忘れる
- **ローカルツールのみ** - 共有またはエンタープライズグレードのツール統合なし  
- **アイデンティティ管理なし** - 特定のユーザーに代わって行動できない

### タスク 1.2: メモリでエージェントを強化する

貴重なお客様が最近の注文に関する問題についてサポートチームに問い合わせます。好みを説明し、不満を伝え、エージェントと協力して問題を解決します。3 週間後、関連する質問について再びサポートに問い合わせます。しかし、エージェントは現在の会話セッションのみを記憶して以前のセッションは記憶しないため、好み、履歴、コンテキストなど、すべてを繰り返す必要があります。これにより、以下が作成されます。
- 情報を繰り返さなければならないことで**不満を抱えた顧客**
- 以前のやり取りを基に構築できない**非効率的なサポート**
- 無機質で一般的な応答による**顧客満足度の低下**

Amazon Bedrock AgentCore Memory は、AI エージェントが長期にわたってコンテキストを維持し、重要な事実を記憶し、一貫性のあるパーソナライズされたエクスペリエンスを実現できるようにするマネージドサービスを提供することで、この制限に対処します。AgentCore Memory は次の 2 つのレベルで動作します。
- **短期メモリ**: 即時の会話コンテキストとセッションベースの情報 (Strands Agent フレームワークによって自動的に処理されます)
- **長期メモリ**: 事実、好み、概要など、複数の会話から抽出された永続的な情報 (USER_PREFERENCE および SEMANTIC 戦略を備えた AgentCore Memory サービスを通じて実装されます)

プロトタイプを、以下の例を実行できる顧客対応アシスタントに変換します。
- **「おかえり、サラ」** - リピーターを即座に認識する
- **「先月のノートパソコンに関する問題のフォローアップ」** - 関連する会話をシームレスにつなげる
- **「購入履歴に基づいたおすすめは次のとおりです」** - パーソナライズされた提案を提供する

このタスクを完了すると、エージェントは統合メモリ機能を備えた次のアーキテクチャを使用できるようになります。

<div style="text-align:left">
    <img src="images/architecture_lab2_memory_ja_jp.png" width="75%"/>
</div>

**画像の説明: 永続的な顧客コンテキストとパーソナライゼーションのための AgentCore Memory で強化されたエージェント**

**メモリストラテジー設定**: 次の 2 つのインテリジェントな戦略を組み合わせてメモリリソースを作成します。

| 戦略タイプ | 目的 | お客様にとってのメリット |
|---------------|---------|------------------|
| USER_PREFERENCE | 顧客の好みと行動 | 「あなたの好みは...」 |
| SEMANTIC | 事実情報とコンテキスト | 「以前の問題について...」 |

AgentCore Memory は、ActorId を使用して長期メモリメッセージを論理的にグループ化するために名前空間を使用します。
- `support/customer/{actorId}/preferences`: ユーザーの好みのメモリ戦略用
- `support/customer/{actorId}/semantic`: セマンティックメモリ戦略用

In [ ]:
# Initialize memory client for AgentCore Memory service
memory_client = MemoryClient(region_name=REGION)
memory_name = "CustomerSupportMemory"

def create_or_get_memory_resource():
    try:
        # Try to get existing memory resource from SSM parameter
        memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
        memory_client.gmcp_client.get_memory(memoryId=memory_id)
        return memory_id
    except:
        # Create new memory resource with two strategies
        strategies = [
            {
                # USER_PREFERENCE strategy captures customer preferences and behaviors
                StrategyType.USER_PREFERENCE.value: {
                    "name": "CustomerPreferences",
                    "description": "Captures customer preferences and behavior",
                    "namespaces": ["support/customer/{actorId}/preferences"],
                }
            },
            {
                # SEMANTIC strategy stores factual information from conversations
                StrategyType.SEMANTIC.value: {
                    "name": "CustomerSupportSemantic",
                    "description": "Stores facts from conversations",
                    "namespaces": ["support/customer/{actorId}/semantic"],
                }
            },
        ]
        print("Creating AgentCore Memory resources (2-3 minutes)...")
        # Create memory resource and wait for completion
        response = memory_client.create_memory_and_wait(
            name=memory_name,
            description="Customer support agent memory",
            strategies=strategies,
            event_expiry_days=90,  # Memory events expire after 90 days
        )
        memory_id = response["id"]
        # Store memory ID in SSM for future use
        put_ssm_parameter("/app/customersupport/agentcore/memory_id", memory_id)
        return memory_id

memory_id = create_or_get_memory_resource()
print(f"✅ Memory resource ready: {memory_id}")

以前にサポートチームとやり取りをしたことがある「customer_001」という名前のリピーターをシミュレーションします。これは、AgentCore Memory が個々の会話を自動的にリッチで永続的なカスタマーインサイトに変換する方法を示しています。以前のお客様とのやり取りをロードして、AgentCore Memory がそれらを自動的に長期的なカスタマーインサイトに変える様子をご覧ください。

In [ ]:
# Seed previous customer interactions
previous_interactions = [
    ("I'm having issues with my MacBook Pro overheating during video editing.", "USER"),
    ("I can help with that thermal issue. Your MacBook Pro order #MB-78432 is still under warranty.", "ASSISTANT"),
    ("What's the return policy on gaming headphones? I need low latency for competitive FPS games", "USER"),
    ("For gaming headphones, you have 30 days to return. Since you're into competitive FPS, I'd recommend checking audio latency specs.", "ASSISTANT"),
    ("I need a laptop under $1200 for programming. Prefer 16GB RAM minimum and good Linux compatibility. I like ThinkPad models.", "USER"),
    ("Perfect! For development work, I'd suggest ThinkPad E series or Dell XPS models with excellent Linux support.", "ASSISTANT"),
]

if memory_id:
    memory_client.create_event(
        memory_id=memory_id,
        actor_id=CUSTOMER_ID,
        session_id="previous_session",
        messages=previous_interactions
    )
    print("✅ Customer history seeded successfully")
    print("⏳ Long-term memory processing will begin automatically...")

Strands Agents は、厳密に型付けされたイベントコールバックを通じてコンポーネントがエージェントの動作に反応したりエージェントの動作を変更したりできるようにする強力なフックシステムを提供します。これにより、手動で操作しなくてもメモリ操作が自動的に行われます。

お客様がエージェントとやり取りするたびに、次の処理が自動的に行われます。
- 以前のやり取りや好みに基づいて**会話をパーソナライズする**
- **新しいやり取りをメモリに追加**して、今後のパーソナライゼーションを継続的に改善する

フック統合が行うこと:
- **応答前**: 関連する顧客コンテキストと好みを自動的に取得する
- **応答後**: 新しいやり取りを AgentCore Memory に自動的に保存する

メモリフックによる自動顧客コンテキストを有効にします。

In [ ]:
class CustomerSupportMemoryHooks(HookProvider):
    def __init__(self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        # Hook that runs before agent responds to retrieve customer context
        messages = event.agent.messages
        # Only process user messages (not tool results)
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                all_context = []
                # Retrieve memories from each strategy namespace, both USER_PREFERENCE and SEMANTIC
                for context_type, namespace in self.namespaces.items():
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,  # Get top 3 relevant memories
                    )
                    # Extract text content from memory objects
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(f"[{context_type.upper()}] {text}")
                
                # Prepend customer context to user message
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = f"Customer Context:\n{context_text}\n\n{original_text}"
            except Exception as e:
                print(f"Failed to retrieve customer context: {e}")

    def save_support_interaction(self, event: AfterInvocationEvent):
        # Hook that runs after agent responds to save interaction to memory
        try:
            messages = event.agent.messages
            # Only save if we have both user and assistant messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                customer_query = None
                agent_response = None
                
                # Find the most recent user query and assistant response
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break
                
                # Save the interaction to AgentCore Memory
                if customer_query and agent_response:
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[(customer_query, "USER"), (agent_response, "ASSISTANT")],
                    )
        except Exception as e:
            print(f"Failed to save support interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        # Register both hooks with the agent's hook registry
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)

print("✅ Memory hooks defined - Automatic customer personalization enabled!")
print("🧠 Your agent will now remember customers and personalize every interaction")

メモリ強化エージェントを作成してテストし、顧客コンテキストを取得して応答をパーソナライズする方法を確認します。

In [ ]:
# Create memory-enhanced agent with hooks
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

memory_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    hooks=[memory_hooks],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Memory-enhanced agent created!")
print("🧠 Agent will automatically retrieve customer context and save interactions")

In [ ]:
# Wait for memory processing to complete
print("⏳ Waiting 90 seconds for memory processing to complete...")
time.sleep(90)

# Test memory recall
print("🧠 Testing memory-enhanced agent...\n")
response = memory_agent("What are my laptop preferences?")
print("\n" + "="*50 + "\n")

### タスク 1.3: ゲートウェイ統合と AgentCore Identity に合わせてスケールする

メモリが用意できたら、強力なツールに焦点を当て、その効影響を拡大します。優れたエージェントには、社内外両方のお客様のために業務を遂行できるように、独自およびサードパーティーの API とデータを最大限に活用できるツールが必要です。しかし、エージェントツールの構築、保護、拡張は難しく、エージェントのプロトタイプから本番環境におけるエージェントの実際のビジネス価値に移行するお客様にとって大きな障害となっています。AgentCore Gateway は、統合**モデルコンテキストプロトコル (MCP)** エンドポイントを通じて AI エージェントが実際のツールを検出、認証、呼び出しできるようにする接続レイヤーとして機能します。これは、何百もの API、リソース、ツールを管理する企業にとって非常に重要です。

主な利点は以下のとおりです。
- インフラストラクチャ管理がない**フルマネージド MCP サーバー**ソリューション
- **既存の API と Lambda 関数の統合**
- 多様なツールでの**統一されたインターフェイス**
- **安全な認証と認可**
- **セマンティックツールの発見**と選択

##### 構築内容:

**ツールの一元化と再利用性:**
- ウェブ検索をローカルツールから集中型の AgentCore Gateway に移行する
- 既存のエンタープライズ Lambda 関数を統合する (保証チェック)
- 複数のエージェントタイプがアクセスできる共有ツールインフラストラクチャを作成する

**エンタープライズグレードのセキュリティ:**
- Cognito 統合を使用した JWT ベース認証を実装する
- ゲートウェイアクセスの安全なインバウンド認可を設定する
- ツールの使用のためのアイデンティティベースのアクセス制御を確立する

これにより、ツールを一元管理して複数のエージェントタイプで再利用できるスケーラブルな基盤が構築され、コードの重複がなくなり、メンテナンスが簡単になります。

AgentCore Identity もこのプロセスに関与しています。これにより、AI エージェントは AWS リソースに安全にアクセスし、Amazon Cognito と連携してインバウンド呼び出し認証に対処する上で役立ちます。また、AI エージェントはアウトバウンド認証を使用してサードパーティーのツールやサービスに安全にアクセスできますが、このラボでは Agentcore Identity のこの機能は使用しません。

<div style="text-align:left">
    <img src="images/architecture_lab3_identity_ja_jp.png" width="75%"/>
</div>

このタスクを完了すると、エージェントは統合ゲートウェイ機能を備えた次のアーキテクチャを使用できるようになります。

<div style="text-align:left">
    <img src="images/architecture_lab3_gateway_ja_jp.png" width="75%"/>
</div>

**画像の説明: 安全で一元的なツール管理とエンタープライズ統合のための、AgentCore Gateway で強化されたエージェント**

AgentCore Gateway を作成し、Lambda 関数を MCP 互換エンドポイントとして公開します。ツールを呼び出す権限のある発信者を検証するには、MCP サーバーの標準である OAuth 認可を使用して**インバウンド認証**を設定します。

### ゲートウェイ認証について理解する

AgentCore Gateway は **OAuth 2.0 と JWT トークン**を使用してツールへのアクセスを保護します。これにより、不正なアプリケーションが Lambda 関数を呼び出すことを防ぎます。

**主要な概念:**

1. **認証プロバイダー**: Amazon Cognito はアイデンティティを管理し、トークンを発行します
2. **クライアント認証情報**: エージェントは client_id と client_secret (アプリケーションのユーザー名 / パスワードなど) を使用します
3. **JWT トークン**: エージェントが承認されていることを証明する短期間のトークン
4. **許可されたクライアント**: Gateway にアクセスできるクライアント ID の許可リスト

**仕組み:**
```
エージェント → Cognito: 「これが私の client_id と client_secret です」
Cognito → エージェント: 「これがあなたの JWT アクセストークンです」
エージェント → ゲートウェイ: 「これが私のトークンです」
ゲートウェイ → Cognito: 「このトークンは有効で、許可されたクライアントからのものですか」
ゲートウェイ → エージェント: 「アクセスが承認されました」
```

**セキュリティ上の注意**: これらの認証情報は事前に作成され、SSM パラメータストアに安全に保存されています。認証情報をコードでハードコーディングすることは決してしないでください

In [ ]:
# Retrieve authentication configuration from SSM Parameter Store
# These values were created by the CloudFormation template

# Client ID: Identifies which application is making the request
machine_client_id = get_ssm_parameter("/app/customersupport/agentcore/machine_client_id")
print(f"Machine Client ID: {machine_client_id}")

# Discovery URL: Tells the Gateway where to find Cognito's OAuth configuration
# This URL provides metadata about token endpoints, supported scopes, etc.
cognito_discovery_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
print(f"Discovery URL: {cognito_discovery_url}")

# Configure JWT-based authentication for the Gateway
auth_config = {
    "customJWTAuthorizer": {
        # Only tokens from this client ID will be accepted
        "allowedClients": [machine_client_id],
        # Gateway will fetch OAuth metadata from this URL
        "discoveryUrl": cognito_discovery_url
    }
}

print("✅ Authentication configuration ready")

### AgentCore Gateway を作成する

Gateway は、エージェントとバックエンド Lambda 関数間の安全なプロキシとして機能します。AI エージェント専用に設計された API Gateway として考えます。

**作成しているもの:**
- **ゲートウェイインフラストラクチャ**: コアゲートウェイリソース
- **MCP プロトコル**: ツール通信のための標準プロトコル
- **JWT 認可**: 作成したばかりの認証設定を使用する
- **IAM ロール**: Lambda 関数を呼び出す権限

**次に起こること:**
1. ゲートウェイを作成する (このセル)
2. ツール定義を含む Lambda ターゲットを追加する (次のセル)
   - 1 つの Lambda 関数で複数のツールを処理する: `check_warranty_status` および `web_search`
   - ゲートウェイ はツール名を Lambda に渡し、Lambda は適切なハンドラーにルーティングする
3. エージェントをゲートウェイに接続する

**注**: CloudFormation テンプレートは複数のツール操作を処理できる 1 つの Lambda 関数 (`CustomerSupportLambda`) をデプロイしました。これはツールごとに個別の Lambda 関数をデプロイするよりも効率的です。

In [ ]:
class CreationFailedError(Exception):
    def __init__(self, message):
        self.message = message
        super().__init__(self.message)

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
gateway_name = "customersupport-gw"

try:
    print(f"Creating gateway: {gateway_name}")
    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role"),
        protocolType="MCP",  # Model Context Protocol
        authorizerType="CUSTOM_JWT",  # Use JWT tokens for auth
        authorizerConfiguration=auth_config,  # Our auth config from above
        description="Customer Support AgentCore Gateway",
    )
    gateway_id = create_response["gatewayId"]
    gateway_url = create_response["gatewayUrl"]
    put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)

    # Wait for Gateway to be ready
    print("Waiting for Gateway to be ready...")
    while True:
        status = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['status']
        if status == 'READY':
            break
        elif status == 'FAILED':
            raise CreationFailedError("Gateway creation failed")
        else:
            print(f"  Status: {status}")
            time.sleep(5)

    print(f"✅ Gateway created successfully!")
    print(f"   Gateway ID: {gateway_id}")
    print(f"   Gateway URL: {gateway_url}")
    
except gateway_client.exceptions.ConflictException:
    # Gateway already exists, retrieve it
    gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    gateway_url = gateway_response["gatewayUrl"]
    print(f"✅ Using existing gateway: {gateway_id}")
    
except CreationFailedError:
    print("\033[31m❌ Gateway creation failed. Check CloudWatch logs for details.\033[0m")

AgentCore Gateway は呼び出すツールの名前を指定して、Lambda コンテキストを設定します。ツールに渡されるパラメータは、Lambda イベントによって指定されます。これにより、既存のエンタープライズ Lambda 関数 (この場合は `AgentCoreLab-CustomerSupportLambda`) を統合して、複数のエージェントで再利用できます。

API 仕様を使用して Lambda 関数をゲートウェイターゲットとして追加します。

In [ ]:
# Load API specification for Lambda tools
api_spec = [
    {
        "name": "check_warranty_status",
        "description": "Check warranty status using serial number and email",
        "inputSchema": {
            "type": "object",
            "properties": {
                "serial_number": {"type": "string"},
                "customer_email": {"type": "string"}
            },
            "required": ["serial_number"]
        }
    },
    {
        "name": "web_search",
        "description": "Search the web for updated information",
        "inputSchema": {
            "type": "object",
            "properties": {
                "keywords": {"type": "string", "description": "Search query keywords"},
                "region": {"type": "string", "description": "Search region (e.g., us-en)"},
                "max_results": {"type": "integer", "description": "Maximum results"}
            },
            "required": ["keywords"]
        }
    }
]

# Create gateway target
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": get_ssm_parameter("/app/customersupport/agentcore/lambda_arn"),
            "toolSchema": {"inlinePayload": api_spec},
        }
    }
}

try:
    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaTarget",
        description="Lambda tools for customer support",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
    )
    print(f"✅ Gateway target created: {create_target_response['targetId']}")
except Exception as e:
    print(f"Gateway target may already exist: {str(e)}")

Cognito の認証トークンを Strands SDK の MCPClient に統合して、安全な MCP 接続を作成します。

ゲートウェイツールにアクセスするための認証済み MCP クライアントを作成します。

In [ ]:
def get_cognito_client_secret():
    # Get Cognito client secret using Cognito API
    client = boto3.client("cognito-idp")
    response = client.describe_user_pool_client(
        UserPoolId=get_ssm_parameter("/app/customersupport/agentcore/userpool_id"),
        ClientId=get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
    )
    return response["UserPoolClient"]["ClientSecret"]

def get_oauth_token():
    # Get OAuth token for gateway authentication using client credentials flow
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    data = {
        "grant_type": "client_credentials",  # OAuth 2.0 client credentials flow
        "client_id": get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
        "client_secret": get_cognito_client_secret(),
        "scope": get_ssm_parameter("/app/customersupport/agentcore/cognito_auth_scope"),
    }
    # Request access token from Cognito
    response = requests.post(
        get_ssm_parameter("/app/customersupport/agentcore/cognito_token_url"),
        headers=headers, data=data
    )
    return response.json()

# Get OAuth access token (JWT format)
token_response = get_oauth_token()
access_token = token_response['access_token']

# Create MCP client with Bearer token authentication
mcp_client = MCPClient(
    url=gateway_url,
    headers={"Authorization": f"Bearer {access_token}"},  # JWT token in Authorization header
)

print(f"✅ MCP client configured for gateway: {gateway_url}")

メモリフック + ローカルツール + ゲートウェイツールすべてを組み合わせます。これにより、一部のツールはローカルのまま (スピードとシンプルさのため)、他のツールはゲートウェイを介して一元化される (再利用性とエンタープライズ統合のため) というハイブリッドアーキテクチャが構築されます。

このアプローチにより、異なるエージェント間でのコードの重複がなくなり、ツール更新の一元管理が可能になります。

In [ ]:
# Initialize memory hooks for customer context
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

# Start MCP client connection to gateway
mcp_client.start()
# Retrieve available tools from the gateway
gateway_tools = mcp_client.list_tools_sync()

# Combine local tools with centralized gateway tools
all_tools = [
    get_product_info,      # Local tool
    get_return_policy,     # Local tool
    get_technical_support, # Local tool
] + gateway_tools          # Gateway tools - web_search, check_warranty_status

# Create enhanced agent with memory and gateway integration
enhanced_agent = Agent(
    model=model,
    tools=all_tools,           # Local + gateway tools
    hooks=[memory_hooks],      # Automatic memory operations
    system_prompt=SYSTEM_PROMPT
)

print("✅ Enhanced Customer Support Agent created!")
print(f"📊 Total tools available: {len(all_tools)}")
print(f"🧠 Memory enabled with ID: {memory_id}")
print(f"🔒 Secure gateway integration: {gateway_url}")

メモリおよびゲートウェイ機能を使用してエージェントをテストします。エージェントがメモリを通じて顧客コンテキストを維持しながらローカルツールと一元化されたゲートウェイツールの両方をシームレスに使用できることを確認します。

テストシナリオには、保証チェック、ウェブ検索、およびメモリ + ゲートウェイ機能の組み合わせ含まれます。

In [ ]:
# Test gateway tools
print("🔍 Testing gateway web search...\n")
response2 = enhanced_agent("Search for latest iPhone 15 troubleshooting tips")
print("\n" + "="*50 + "\n")

In [ ]:
# Test warranty check
print("🛡️ Testing warranty check...\n")
response3 = enhanced_agent("Check warranty status for serial number ABC12345678")
print("\n" + "="*50 + "\n")

In [ ]:
# Test combined capabilities
print("🎯 Testing combined memory + gateway capabilities...\n")
response4 = enhanced_agent("I need gaming headphones again, and also search for the latest reviews")
print("\n" + "="*50 + "\n")

## 次のステップ

🎉 **お疲れ様でした。** ノートブックの演習が完了しました。

以下の作業が完了しました。
- Strands を使用して基本的な AI エージェントプロトタイプを作成した
- AgentCore Memory を使用して拡張し、顧客コンテキストを永続的に把握できるようにした
- 安全で一元的なツール共有のための統合型 AgentCore Gateway
- 本番環境ですぐに使用できるカスタマーサポートシステム全体をテストした

### 次のステップ

1. **このノートブックファイルを閉じます**
2. **ラボの手順に戻ります**
3. **タスク 2 に進みます**。AgentCore ダッシュボードを調べて、リソースを実際に確認します。
